# NTL 0627 Small Encoder Ensemble

0.627 plus 10% UltraBERT/ELECTRA mixture. Main inference follows our scored 0.627 reproduction,
submission 57564496, with Nishant part two and Nuri's pretrained encoders.
Nuri republished the encoder package as ntl-two-stage-encoders; every public
run computes all six models and checks their probabilities against our scored
three-row public reference. The reference is used only for validation.

Only this notebook's named addition is enabled. T4 x2, Internet OFF; no Qwen,
LoRA, or encoder fine-tuning. ELECTRA uses a small CPU-fitted readout. Gaussian
statistics and that readout come from a one-time, source-checked preparation run.


In [ ]:
_NTL_EXTENSION_MODULES = {'ntl_encoder_models.py': '"""Real small-encoder inference and the existing public ATLAS checkpoint."""\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\nimport numpy as np\n\n\ndef softmax(scores):\n    values = np.asarray(scores, dtype=np.float64)\n    values = np.exp(values - values.max(1, keepdims=True))\n    return values / values.sum(1, keepdims=True)\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef one(root, pattern):\n    hits = sorted(Path(root).rglob(pattern))\n    if len(hits) != 1:\n        raise ValueError(f\'Expected one input {pattern}; found {len(hits)}\')\n    return hits[0]\n\n\ndef train_fingerprint(frame):\n    rows = frame[[\'id\', \'text\', \'model\', \'prompt_id\']].astype(str).to_dict(\'records\')\n    return hashlib.sha256(json.dumps(rows, sort_keys=True, ensure_ascii=False, separators=(\',\', \':\')).encode()).hexdigest()\n\n\ndef load_cache(namespace, artifact):\n    manifest_path = one(namespace[\'INPUT_DIR\'], \'ntl0627_cache/manifest.json\')\n    manifest = json.loads(manifest_path.read_text())\n    if manifest[\'schema\'] != \'ntl0627-encoder-cache-v1\' or not manifest[\'passed\']:\n        raise ValueError(\'The feature/readout cache did not pass preparation checks\')\n    if manifest[\'labels\'] != namespace[\'LABELS\'] or manifest[\'train_fingerprint\'] != train_fingerprint(namespace[\'train\']):\n        raise ValueError(\'Cache train rows, text, labels or label order differ\')\n    for package in [\'numpy\', \'scikit-learn\', \'torch\', \'transformers\']:\n        if manifest[\'versions\'][package] != importlib.metadata.version(package):\n            raise ValueError(f\'Feature-cache runtime version differs: {package}\')\n    path = manifest_path.parent / artifact\n    if sha(path) != manifest[\'artifacts\'][artifact][\'sha256\']:\n        raise ValueError(f\'Cache artifact checksum differs: {artifact}\')\n    return manifest, np.load(path, allow_pickle=False)\n\n\ndef load_checked_model(auto_class, path, **kwargs):\n    model, information = auto_class.from_pretrained(path, output_loading_info=True, **kwargs)\n    parameters = set(dict(model.named_parameters()))\n    missing = parameters.intersection(information.get(\'missing_keys\', []))\n    if missing or information.get(\'mismatched_keys\'):\n        raise ValueError(\'A requested pretrained model has missing or mismatched learned parameters\')\n    return model\n\n\ndef small_encoder_probabilities(namespace):\n    import torch\n    from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification\n    from ntl_staging import stage_checkpoint\n    from ntl_encoder_pooling import pooled_states\n\n    root = namespace[\'INPUT_DIR\']\n    labels = namespace[\'LABELS\']\n    label_to_index = {label: index for index, label in enumerate(labels)}\n    texts = namespace[\'test\'].text.fillna(\'\').astype(str).tolist()\n    manifest, head = load_cache(namespace, \'electra_head.npz\')\n    path = one(root, \'electra-large-discriminator/config.json\').parent\n    path = stage_checkpoint(path)\n    config = json.loads((path / \'config.json\').read_text())\n    if config.get(\'model_type\') != \'electra\' or config.get(\'hidden_size\') != 1024:\n        raise ValueError(\'Wrong ELECTRA backbone architecture\')\n    tokenizer = AutoTokenizer.from_pretrained(str(path))\n    model = load_checked_model(AutoModel, str(path), dtype=torch.float32).cuda().eval()\n    train = namespace[\'train\']\n    if head[\'anchor_ids\'].astype(str).tolist() != train.id.astype(str).tolist()[:3]:\n        raise ValueError(\'ELECTRA anchor IDs differ\')\n    combined = train.text.fillna(\'\').astype(str).tolist()[:3] + texts\n    with torch.inference_mode():\n        parts = pooled_states(model, tokenizer, combined, [(21, \'mean\'), (24, \'style\')], False, 16)\n    features = np.concatenate([parts[\'L21_mean\'], parts[\'L24_style\']], axis=1)\n    relative = float(np.abs(features[:3] - head[\'anchor_features\']).mean() /\n                     max(np.abs(head[\'anchor_features\']).mean(), 1e-9))\n    if relative >= .02:\n        raise ValueError(\'ELECTRA forward pass does not match the public cached train features\')\n    if head[\'classes\'].tolist() != list(range(18)):\n        raise ValueError(\'ELECTRA readout class order differs\')\n    logits = ((features[3:].astype(np.float64) - head[\'mean\']) / head[\'scale\']) @ head[\'coef\'].T + head[\'bias\']\n    electra = softmax(logits)\n    del model, tokenizer, parts, features, head\n    gc.collect()\n    torch.cuda.empty_cache()\n\n    # Five genuinely fine-tuned UltraBERT heads, averaged in the declared label order.\n    configs = sorted(Path(root).rglob(\'fold*/config.json\'))\n    folds = [p.parent for p in configs if json.loads(p.read_text()).get(\'model_type\') == \'ultrabert\']\n    if len(folds) != 5:\n        raise ValueError(\'Expected all five pretrained UltraBERT folds\')\n    ultra = np.zeros((len(texts), 18), dtype=np.float64)\n    fold_records = []\n    for original in folds:\n        path = stage_checkpoint(original)\n        tokenizer = AutoTokenizer.from_pretrained(str(path), trust_remote_code=True)\n        model = load_checked_model(AutoModelForSequenceClassification, str(path), trust_remote_code=True,\n                                   dtype=torch.float16, attn_implementation=\'sdpa\').cuda().eval()\n        model_labels = [model.config.id2label[i] for i in range(model.config.num_labels)]\n        if len(model_labels) != 18 or set(model_labels) != set(labels):\n            raise ValueError(\'UltraBERT does not contain the 18 competition labels\')\n        columns = [label_to_index[label] for label in model_labels]\n        ids = tokenizer(texts, truncation=True, max_length=512)[\'input_ids\']\n        order = np.argsort([len(tokens) for tokens in ids], kind=\'stable\')\n        with torch.inference_mode():\n            for start in range(0, len(order), 32):\n                index = order[start:start + 32]\n                batch = tokenizer.pad({\'input_ids\': [ids[i] for i in index]}, return_tensors=\'pt\').to(\'cuda\')\n                values = model(**batch).logits.float().softmax(-1).cpu().numpy()\n                if not np.isfinite(values).all():\n                    raise ValueError(\'Nonfinite UltraBERT probabilities\')\n                ultra[np.ix_(index, columns)] += values / 5\n        fold_records.append(original.name)\n        del model, tokenizer\n        gc.collect()\n        torch.cuda.empty_cache()\n    for values in [ultra, electra]:\n        if not np.isfinite(values).all() or not np.allclose(values.sum(1), 1, atol=1e-5):\n            raise ValueError(\'Invalid small-encoder output probabilities\')\n    mixed = .75 * ultra + .25 * electra\n    return mixed, {\'method\': \'ultrabert75-electra25\', \'ultrabert_folds\': fold_records,\n                   \'electra_anchor_mean_relative_error\': relative, \'electra_head_fit_on_cpu\': True,\n                   \'neural_fine_tuning\': False, \'normalization\': \'ELECTRA train-fitted scaler\',\n                   \'rows\': len(texts)}, {\'ultrabert\': ultra, \'electra\': electra}\n\n\ndef atlas_probabilities(namespace, rows, deadline):\n    import torch\n    from safetensors.torch import load_file\n    from ntl_staging import stage_checkpoint\n\n    root = namespace[\'INPUT_DIR\']\n    labels = namespace[\'LABELS\']\n    bundle = stage_checkpoint(one(root, \'atlas_bundle.json\').parent)\n    sys.path.insert(0, str(bundle))\n    try:\n        from ntl_atlas.model import load_model\n        from ntl_rlcd.data import LABELS as atlas_labels\n    finally:\n        sys.path.remove(str(bundle))\n    cfg = json.loads((bundle / \'atlas_bundle.json\').read_text())[\'training_config\']\n    configs = []\n    for path in Path(root).rglob(\'config.json\'):\n        config = json.loads(path.read_text())\n        if config.get(\'model_type\') == \'qwen3\' and config.get(\'hidden_size\') == 4096 and config.get(\'num_hidden_layers\') == 36:\n            if (path.parent / \'model.safetensors.index.json\').exists() or (path.parent / \'model.safetensors\').exists():\n                configs.append(path.parent)\n    if len(configs) != 1:\n        raise ValueError(\'Expected the pretrained Qwen3-8B-Base for ATLAS\')\n    model, tokenizer = load_model(str(configs[0]), atlas_labels, cfg, device=\'auto\')\n    try:\n        model.load_trainable(load_file(str(bundle / \'adapters.safetensors\')))\n        model.eval()\n        if set(atlas_labels) != set(labels):\n            raise ValueError(\'ATLAS class labels differ\')\n        texts = namespace[\'test\'].text.values[rows].tolist()\n        tokens = tokenizer(texts, truncation=True, max_length=cfg[\'max_length\'])[\'input_ids\']\n        order = np.argsort([len(value) for value in tokens], kind=\'stable\')\n        output = np.zeros((len(rows), 18), dtype=np.float32)\n        cumulative_tokens = np.cumsum([len(tokens[int(i)]) for i in order])\n        began = time.monotonic()\n        for start in range(0, len(order), 4):\n            index = order[start:start + 4]\n            batch = tokenizer.pad({\'input_ids\': [tokens[int(i)] for i in index]}, padding=True, return_tensors=\'pt\').to(model.device)\n            with torch.inference_mode():\n                probabilities, _ = model.score(batch[\'input_ids\'], batch[\'attention_mask\'], source_chunk=3)\n            output[index] = probabilities.float().cpu().numpy()\n            done = start + len(index)\n            if done > 50:\n                elapsed = time.monotonic() - began\n                projected = time.monotonic() + elapsed / cumulative_tokens[done - 1] * (cumulative_tokens[-1] - cumulative_tokens[done - 1]) * 1.1\n                if projected > deadline:\n                    raise RuntimeError(\'Required ATLAS rows would exceed the inference deadline\')\n            if time.monotonic() > deadline:\n                raise RuntimeError(\'ATLAS inference deadline reached\')\n        if not np.isfinite(output).all() or not np.allclose(output.sum(1), 1, atol=.0001):\n            raise ValueError(\'Invalid ATLAS probabilities\')\n        return output[:, [atlas_labels.index(label) for label in labels]]\n    finally:\n        del model\n        gc.collect()\n        torch.cuda.empty_cache()\n', 'ntl_encoder_pooling.py': '"""Frozen ELECTRA pooling copied from the credited public complementary stack.\n\nSource: prvsiyan / Lucas Boesen / kirilltym, retained in Joseph Adamski\'s\nTeach the Model notebook and our scored 0.608 source. Model inference only.\n"""\nimport re\nimport numpy as np\nfrom sklearn.feature_extraction.text import ENGLISH_STOP_WORDS\ntry:\n    import torch\nexcept ImportError:\n    torch = None\n\ndef style_token_mask(tok):\n    """Vocabulary mask of \'style\' tokens: the decoded piece is an English stop word or pure punctuation."""\n    pieces = [tok.decode([i]) for i in range(len(tok))]\n    return torch.tensor([p.strip().lower() in ENGLISH_STOP_WORDS or bool(re.fullmatch(\'\\\\s*[^\\\\w\\\\s]+\\\\s*\', p)) for p in pieces])\n\ndef pooled_states(model, tok, texts, parts, decoder, bs, max_len=512):\n    """{\'L{layer}_{pool}\': (n, hidden)} masked mean pools of hidden_states[layer] (all tokens, or style tokens only)."""\n    H = model.config.hidden_size\n    out = {f\'L{L}_{pool}\': np.zeros((len(texts), H), dtype=np.float32) for L, pool in parts}\n    smask = style_token_mask(tok) if any((pool == \'style\' for _, pool in parts)) else None\n    dev = next(model.parameters()).device\n    order = np.argsort([len(t) for t in texts])\n    retried = 0\n    for b0 in range(0, len(texts), bs):\n        idx = order[b0:b0 + bs]\n        if decoder:\n            batch = [(tok.bos_token or \'\') + texts[i] for i in idx]\n            enc = tok(batch, return_tensors=\'pt\', padding=True, truncation=True, max_length=max_len, add_special_tokens=False).to(dev)\n        else:\n            enc = tok([texts[i] for i in idx], return_tensors=\'pt\', padding=True, truncation=True, max_length=max_len).to(dev)\n        hs = model(**enc, output_hidden_states=True).hidden_states\n        if decoder and (not all((torch.isfinite(hs[L]).all() for L, _ in parts))):\n            retried += 1\n            model.to(torch.bfloat16)\n            hs = model(**enc, output_hidden_states=True).hidden_states\n        m_all = enc[\'attention_mask\'].float()\n        for L, pool in parts:\n            m = m_all * smask.to(m_all.device)[enc[\'input_ids\']].float() if pool == \'style\' else m_all\n            m = m.unsqueeze(-1).to(hs[L].device)\n            out[f\'L{L}_{pool}\'][idx] = ((hs[L].float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()\n        del hs, enc\n    if retried:\n        print(f\'  {retried} batches redone in bf16\', flush=True)\n    return {k: np.nan_to_num(v) for k, v in out.items()}\n', 'ntl_encoder_experiments.py': '"""Four isolated additions to the actual six-member 0.627 baseline."""\nimport json\nfrom pathlib import Path\nimport re\nimport time\n\nimport numpy as np\nfrom scipy.cluster.hierarchy import fcluster, linkage\nfrom scipy.spatial.distance import squareform\nfrom sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer\n\n\ndef normalized_probabilities(values):\n    result = np.asarray(values, dtype=np.float64)\n    if result.ndim != 2 or result.shape[1] != 18 or not np.isfinite(result).all() or (result < 0).any():\n        raise ValueError(\'Expected finite 18-class probabilities\')\n    if not np.allclose(result.sum(1), 1, atol=1e-5):\n        raise ValueError(\'Probability rows do not sum to one\')\n    return result / result.sum(1, keepdims=True)\n\n\ndef mix_probabilities(base, added, weight, rows=None):\n    base = normalized_probabilities(base)\n    if not 0 <= weight <= 1:\n        raise ValueError(\'Mixture weight must be between zero and one\')\n    result = base.copy()\n    rows = np.arange(len(base)) if rows is None else np.asarray(rows)\n    added = normalized_probabilities(added)\n    if added.shape != result[rows].shape:\n        raise ValueError(\'Mixture rows or class count differ\')\n    result[rows] = (1 - weight) * result[rows] + weight * added\n    return normalized_probabilities(result)\n\n\ndef consensus_groups(texts, word_groups):\n    """Intersect independent word and content-character partitions."""\n    rows = len(texts)\n    words = np.asarray(word_groups)\n    if words.shape != (rows,):\n        raise ValueError(\'Invalid word partition\')\n    if rows < 2:\n        return np.arange(rows), np.zeros(rows, bool), np.arange(rows)\n    cleaned = [\' \'.join(word for word in re.findall(r\'[a-z]+\', text.lower()) if word not in ENGLISH_STOP_WORDS) for text in texts]\n    try:\n        vectors = TfidfVectorizer(analyzer=\'char\', ngram_range=(3, 5), sublinear_tf=True, max_features=100000).fit_transform(cleaned)\n    except ValueError as error:\n        if \'empty vocabulary\' not in str(error):\n            raise\n        return np.arange(rows), np.zeros(rows, bool), np.arange(rows)\n    distance = np.clip(1 - (vectors @ vectors.T).toarray(), 0, 1)\n    np.fill_diagonal(distance, 0)\n    characters = fcluster(linkage(squareform(distance, checks=False), method=\'average\'), .55, criterion=\'distance\')\n    _, a, counts_a = np.unique(words, return_inverse=True, return_counts=True)\n    _, b, counts_b = np.unique(characters, return_inverse=True, return_counts=True)\n    _, intersections = np.unique(np.column_stack([a, b]), axis=0, return_inverse=True)\n    safe_groups = np.arange(rows)\n    eligible = np.zeros(rows, dtype=bool)\n    for group in np.unique(intersections):\n        index = np.flatnonzero(intersections == group)\n        count = len(index)\n        size_a, size_b = counts_a[a[index[0]]], counts_b[b[index[0]]]\n        confidence = count / (size_a + size_b - count)\n        if 2 <= count <= 18 and max(size_a, size_b) <= 18 and confidence >= .75:\n            safe_groups[index] = rows + int(group)\n            eligible[index] = True\n    return np.unique(safe_groups, return_inverse=True)[1], eligible, characters\n\n\ndef public_baseline_parity(namespace):\n    reference = namespace[\'_NTL_PUBLIC_REFERENCE\']\n    ids = namespace[\'test\'].id.astype(str).tolist()\n    if ids != reference[\'ids\']:\n        return {\'status\': \'not_public_fixture\', \'prediction_cache_used\': False}\n    if namespace[\'LABELS\'] != reference[\'labels\']:\n        raise ValueError(\'Public reference label order differs\')\n    difference = np.abs(namespace[\'P\'] - np.asarray(reference[\'probabilities\']))\n    result = {\'status\': \'passed\', \'max_absolute_error\': float(difference.max()),\n              \'mean_absolute_error\': float(difference.mean()), \'prediction_cache_used\': False}\n    if result[\'max_absolute_error\'] > .005 or result[\'mean_absolute_error\'] > .0005:\n        raise ValueError(\'The relocated encoder checkpoint does not reproduce the scored public probabilities\')\n    return result\n\n\ndef install_gaussian_hook(namespace):\n    from ntl_encoder_models import load_cache, one, sha\n    from ntl_gaussian import scores\n\n    manifest, arrays = load_cache(namespace, \'gaussian_qwen14.npz\')\n    parameters = {key: arrays[key] for key in arrays.files}\n    arrays.close()\n    metadata = manifest[\'gaussian\']\n    parameters.update(cfg=metadata[\'cfg\'], T1=metadata[\'T1\'], Tg=metadata[\'Tg\'])\n    checkpoint = one(namespace[\'INPUT_DIR\'], \'lora_qwen3_14b/meta.json\').parent\n    for relative, key in [(\'meta.json\', \'checkpoint_meta_sha256\'), (\'adapter/adapter_model.safetensors\', \'adapter_sha256\'), (\'heads.safetensors\', \'heads_sha256\')]:\n        if sha(checkpoint / relative) != metadata[key]:\n            raise ValueError(\'Gaussian cache was fitted for different LoRA weights or metadata\')\n    original = namespace[\'lora_features\']\n\n    def capture(ckpt_dir, base_dir, texts, bs):\n        features, heads = original(ckpt_dir, base_dir, texts, bs)\n        if Path(ckpt_dir).name == \'lora_qwen3_14b\':\n            if sha(Path(base_dir) / \'config.json\') != metadata[\'base_config_sha256\'] or sha(Path(base_dir) / \'tokenizer_config.json\') != metadata[\'tokenizer_config_sha256\']:\n                raise ValueError(\'Gaussian base config or tokenizer differs\')\n            groups, rows, _ = consensus_groups(texts, namespace[\'groups\'])\n            probabilities = np.exp(scores(features, groups, parameters))\n            namespace[\'_NTL_GAUSSIAN_PROB\'] = normalized_probabilities(probabilities)\n            namespace[\'_NTL_GAUSSIAN_ROWS\'] = np.flatnonzero(rows)\n            namespace[\'_NTL_GAUSSIAN_REPORT\'] = {\'cache_used\': True, \'rows_scored\': len(texts),\n                \'rows_mixed\': int(rows.sum()), \'group_views\': [\'word_content\', \'character_content\'],\n                \'cfg\': parameters[\'cfg\'], \'temperatures\': [parameters[\'T1\'], parameters[\'Tg\']],\n                \'extra_Qwen_test_forward\': False, \'neural_training\': False}\n        return features, heads\n\n    namespace[\'lora_features\'] = capture\n\n\ndef run_variant(namespace, config):\n    from ntl_encoder_models import atlas_probabilities, small_encoder_probabilities\n    from ntl_soft_mrf import soft_constraint_decode\n\n    baseline = normalized_probabilities(namespace[\'P\'])\n    baseline_parity = public_baseline_parity(namespace)\n    raw = normalized_probabilities(namespace[\'softmax\'](namespace[\'LOGIT\']))\n    rows = len(raw)\n    kind = config[\'kind\']\n    extras = {}\n    if kind == \'atlas\':\n        margins = np.sort(raw, axis=1)\n        margins = margins[:, -1] - margins[:, -2]\n        selected = np.flatnonzero(margins < .30) if rows >= 50 else np.arange(rows)\n        if len(selected):\n            remaining = max(0, 31800 - (time.time() - namespace[\'T0\']))\n            values = atlas_probabilities(namespace, selected, time.monotonic() + remaining)\n            result = mix_probabilities(raw, values, .10, selected)\n        else:\n            result = raw.copy()\n        extras = {\'weight\': .10, \'margin_threshold\': .30, \'rows\': len(selected),\n                  \'all_selected_rows_completed\': True, \'neural_training\': False}\n    elif kind == \'encoders\':\n        values, extras, members = small_encoder_probabilities(namespace)\n        result = mix_probabilities(raw, values, .10)\n        np.savez_compressed(\'small-encoder-probabilities.npz\', ids=namespace[\'test\'].id.to_numpy(dtype=str),\n                            labels=np.asarray(namespace[\'LABELS\']), **members)\n        extras[\'outer_weight\'] = .10\n    elif kind == \'gaussian\':\n        selected = namespace.get(\'_NTL_GAUSSIAN_ROWS\')\n        values = namespace.get(\'_NTL_GAUSSIAN_PROB\')\n        if selected is None or values is None:\n            raise ValueError(\'The Gaussian head did not run on the actual Qwen features\')\n        result = mix_probabilities(raw, values[selected], .10, selected) if len(selected) else raw.copy()\n        extras = dict(namespace[\'_NTL_GAUSSIAN_REPORT\'], weight=.10)\n    elif kind == \'mrf\':\n        _, _, characters = consensus_groups(namespace[\'test\'].text.astype(str).tolist(), namespace[\'groups\'])\n        selected_labels, extras = soft_constraint_decode(baseline, namespace[\'groups\'], characters,\n            {\'penalty_strength\': .30, \'min_group_confidence\': .75, \'max_mean_log_loss\': .10, \'max_assigned_rank\': 2})\n        extras[\'group_views\'] = [\'word_content\', \'character_content\']\n        result = baseline.copy()\n    else:\n        raise ValueError(\'Unknown encoder experiment\')\n    if kind != \'mrf\':\n        result = namespace[\'sinkhorn\'](result) if rows >= 50 else result\n        selected_labels = result.argmax(1)\n    result = normalized_probabilities(result)\n    final = namespace[\'sub\'].copy()\n    final[\'model\'] = np.asarray(namespace[\'LABELS\'])[selected_labels]\n    if final.id.tolist() != namespace[\'test\'].id.astype(str).tolist() or not final.id.is_unique or not final.model.isin(namespace[\'LABELS\']).all():\n        raise ValueError(\'Final variant submission IDs or labels differ\')\n    final.to_parquet(\'submission.parquet\', index=False)\n    np.savez_compressed(\'variant-probabilities.npz\', probabilities=result, baseline_probabilities=baseline,\n        predicted_class_indices=selected_labels, ids=final.id.to_numpy(dtype=str), labels=np.asarray(namespace[\'LABELS\']))\n    report = {\'pipeline\': \'ntl0627-four-variants-v1\', \'passed\': True, \'config\': config, \'rows\': rows,\n              \'ids\': final.id.tolist(), \'labels\': namespace[\'LABELS\'],\n              \'baseline_six_members_completed\': True, \'baseline_public_parity\': baseline_parity,\n              \'required_addition_completed\': True, \'addition\': extras, \'neural_training\': False,\n              \'seconds\': time.time() - namespace[\'T0\'], \'score\': None}\n    Path(\'variant-report.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    namespace[\'P\'], namespace[\'sub\'] = result, final\n    print(\'NTL0627_VARIANT_VALIDATED\', kind, json.dumps({\'rows\': rows, \'public_parity\': baseline_parity[\'status\']}), flush=True)\n    return report\n', 'ntl_staging.py': '"""Sequentially stage a small immutable checkpoint, preserving every file byte.\n\nIntended for the slow mounted Ultrabert fold checkpoints. No weight changes;\nsource/destination hashes are recorded. Uses scratch outside saved outputs.\n"""\nfrom pathlib import Path\nimport hashlib,json,shutil,time\n\ndef stage_named_files(source,names):\n    """Copy selected flattened public files to scratch under local aliases."""\n    source=Path(source).resolve();assert source.is_relative_to(Path(\'/kaggle/input\').resolve())\n    for origin,alias in names.items():\n        assert Path(origin).name==origin and Path(alias).name==alias\n        assert (source/origin).is_file()\n    key=hashlib.sha256((str(source)+json.dumps(names,sort_keys=True)).encode()).hexdigest()[:16]\n    target=Path(\'/kaggle/temp/ntl-selected-checkpoints\')/key;target.mkdir(parents=True,exist_ok=True)\n    ready=target/\'stage-audit.json\'\n    if not ready.exists():\n        total=sum((source/name).stat().st_size for name in names)\n        assert shutil.disk_usage(target).free>total+2**30\n        records={};began=time.monotonic()\n        for name,alias in names.items():\n            digest=hashlib.sha256();count=0\n            with (source/name).open(\'rb\') as src,(target/alias).open(\'wb\') as dst:\n                while chunk:=src.read(16*1024*1024):dst.write(chunk);digest.update(chunk);count+=len(chunk)\n            assert count==(source/name).stat().st_size\n            with (target/alias).open(\'rb\') as check:assert hashlib.file_digest(check,\'sha256\').hexdigest()==digest.hexdigest()\n            records[name]=dict(local_alias=alias,bytes=count,sha256=digest.hexdigest())\n        ready.write_text(json.dumps(dict(source=str(source),files=records,seconds=time.monotonic()-began),indent=2))\n        print(\'VERIFIED_SELECTED_CHECKPOINT_STAGE\',total,round(time.monotonic()-began,2),flush=True)\n    audit_dir=Path(\'/kaggle/working/model_stage_audits\');audit_dir.mkdir(exist_ok=True)\n    (audit_dir/(key+\'.json\')).write_bytes(ready.read_bytes())\n    return target\n\ndef stage_checkpoint(source,max_bytes=3*2**30):\n    source=Path(source).resolve()\n    assert source.is_relative_to(Path(\'/kaggle/input\').resolve())\n    files=[p for p in source.rglob(\'*\') if p.is_file()]\n    total=sum(p.stat().st_size for p in files)\n    if total>max_bytes:\n        print(\'CHECKPOINT_STAGE_SIZE_GUARD\',str(source),total,flush=True);return source\n    target=Path(\'/kaggle/temp/ntl-small-checkpoints\')/hashlib.sha256(str(source).encode()).hexdigest()[:16]\n    ready=target/\'stage-audit.json\'\n    if ready.exists():\n        audit_dir=Path(\'/kaggle/working/model_stage_audits\');audit_dir.mkdir(exist_ok=True)\n        (audit_dir/(target.name+\'.json\')).write_bytes(ready.read_bytes())\n        return target\n    target.mkdir(parents=True,exist_ok=True)\n    if shutil.disk_usage(target).free<total+2**30:\n        print(\'CHECKPOINT_STAGE_DISK_GUARD\',str(source),flush=True);return source\n    began=time.monotonic();records={}\n    try:\n        for origin in files:\n            relative=origin.relative_to(source);destination=target/relative;destination.parent.mkdir(parents=True,exist_ok=True);digest=hashlib.sha256();count=0\n            with origin.open(\'rb\') as src,destination.open(\'wb\') as dst:\n                while chunk:=src.read(16*1024*1024):dst.write(chunk);digest.update(chunk);count+=len(chunk)\n            assert count==origin.stat().st_size\n            with destination.open(\'rb\') as dst:assert hashlib.file_digest(dst,\'sha256\').hexdigest()==digest.hexdigest()\n            records[str(relative)]={\'bytes\':count,\'sha256\':digest.hexdigest()}\n        ready.write_text(json.dumps(dict(source=str(source),files=records,seconds=time.monotonic()-began),indent=2))\n        audit_dir=Path(\'/kaggle/working/model_stage_audits\');audit_dir.mkdir(exist_ok=True)\n        (audit_dir/(target.name+\'.json\')).write_bytes(ready.read_bytes())\n        print(\'VERIFIED_LOCAL_CHECKPOINT_STAGE\',str(source),total,round(time.monotonic()-began,2),flush=True);return target\n    except Exception as exc:\n        print(\'CHECKPOINT_STAGE_FALLBACK\',str(source),type(exc).__name__,flush=True);return source\n', 'ntl_gaussian.py': '"""Shared-prompt Gaussian group head (ported, Apache-2.0, from public Kaggle notebooks by prvsiyan / kirilltym / sangam21).\nTwo-way model x = class mean + prompt effect + noise; exact (n<=3) or cavity-EM (n>3) injective group marginals."""\nimport itertools\nimport numpy as np\nK = 18\nDEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,\n           "iters": 5, "repel": True, "sp_scale": 1.0}\n\n# =================================================================== primitives ====\ndef _lse(A, axis=None, keepdims=False):\n    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)\n    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m\n    if keepdims: return out\n    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())\n\ndef log_softmax(L): return L - _lse(L, axis=1, keepdims=True)\n\ndef _lw_shrinkage(X):\n    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""\n    from sklearn.covariance import ledoit_wolf_shrinkage\n    return float(ledoit_wolf_shrinkage(X, assume_centered=True))\n\ndef _shrunk(S, a):\n    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)\n\ndef _two_way(Z, y, pinv, G, iters=4):\n    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""\n    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)\n    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))\n    for _ in range(iters):\n        R = Z - P[pinv]\n        for c in cls: mu[c] = R[y == c].mean(0)\n        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]\n    return mu, P, cnt_p\n\ndef fit(X, y, prompts, cfg=None):\n    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.\n    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller\'s cross-fit)."""\n    c = dict(DEFAULT); c.update(cfg or {})\n    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape\n    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)\n    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx\n    if c["red"] == "none":\n        V = np.eye(D)\n    else:\n        if c["red"] == "wpca":\n            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]\n        elif c["red"] == "pca": R = Xs\n        else: raise ValueError(c["red"])\n        d = int(min(c["d"], N - G - 1, D))\n        # economy SVD through the (N x N) Gram matrix (N << D for the real views)\n        if N < D:\n            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]\n            V = (R.T @ U) / np.sqrt(w)\n        else:\n            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T\n    Z = Xs @ V; d = Z.shape[1]\n    mu, P, ng = _two_way(Z, y, pinv, G)\n    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)\n    Se_raw = E.T @ E / dof\n    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)\n    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw\n    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])\n    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]\n    # joint diagonalisation: A Se A\' = I, A Sp A\' = diag(lam)\n    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)\n    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li\n    W = V @ A.T\n    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,\n            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}\n\ndef transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]\n\n_MASK = {}\ndef _inj_mask(n, C):\n    if (n, C) not in _MASK:\n        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)\n        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]\n        _MASK[(n, C)] = bad\n    return _MASK[(n, C)]\n\ndef singleton_ll(U, m, lam):\n    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""\n    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)\n\ndef joint_terms(U, m, lam):\n    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""\n    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]\n    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)\n    return a, B\n\ndef joint_table(a, B):\n    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""\n    n, C = a.shape; Q = np.zeros([C] * n)\n    for i in range(n):\n        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)\n    for i, j in itertools.combinations(range(n), 2):\n        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)\n    return Q\n\ndef exact_marginals(Q, T=1.0, inj=True):\n    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""\n    n = Q.ndim; C = Q.shape[0]; Qt = Q / T\n    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)\n    lz = _lse(Qt); out = np.empty((n, C))\n    for i in range(n):\n        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz\n    return out\n\ndef em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):\n    """cavity EM for one group (any n >= 2): returns (n, C) log q."""\n    n = len(U); v = lam / (1 + (n - 1) * lam)\n    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))\n    q = np.exp(log_softmax(L))\n    for _ in range(iters):\n        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)\n        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T\n        if repel:   # mean-field at-most-once: discourage classes the other members already hold\n            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)\n        q = np.exp(log_softmax(L))\n    return log_softmax(L)\n\nLAM_DIFF = 1e6\ndef lam_g(prm, cfg=None):\n    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg \'diff\'): then the joint likelihood\n    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs\n    centred prototypes) and the prompt term cancels exactly."""\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})\n    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]\n\ndef group_logpost(U, prm, T1=None, Tg=None, cfg=None):\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)\n    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg\n    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)\n    if n <= c["n_exact"]:\n        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])\n    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])\n\ndef scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):\n    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""\n    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))\n    sizes = np.bincount(lab); s1 = sizes[lab] == 1\n    T1 = prm["T1"] if T1 is None else T1\n    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)\n    for g in np.where(sizes >= 2)[0]:\n        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)\n    return out\n', 'ntl_soft_mrf.py': '"""MRF-inspired finite cannot-link penalties on the final ensemble probabilities.\n\nThis is an adaptation of Basu, Bilenko and Mooney (KDD 2004), not a reproduction\nof HMRF-KMEANS. The graph is a set of disjoint weighted cliques formed by the\nintersections of two existing prompt partitions. A uniform pair penalty inside\neach clique permits exact MAP decoding with repeated label slots.\n"""\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\n\n\nSOFT_MRF_DEFAULTS = {\n    \'penalty_strength\': 0.40,\n    \'min_group_size\': 2,\n    \'max_parent_size\': 18,\n    \'min_group_confidence\': 0.50,\n    \'min_energy_improvement\': 0.05,\n    \'min_configuration_gap\': 0.05,\n    \'min_assigned_probability\': 0.01,\n    \'min_relative_probability\': 1.0 / 3.0,\n    \'max_mean_log_loss\': 0.15,\n    \'max_assigned_rank\': 3,\n}\n\n\ndef _soft_energy(unary, labels, penalty):\n    counts = np.bincount(labels, minlength=unary.shape[1])\n    return float(unary[np.arange(len(labels)), labels].sum()\n                 + penalty * (counts * (counts - 1) / 2).sum())\n\n\ndef _solve_soft_clique(unary, penalty):\n    """Exact optimum and next distinct *label configuration*, not slot permutation.\n\nGiving the j-th use of label c a cost penalty*j realizes the total pair penalty\npenalty*count_c*(count_c-1)/2. Every row can use every label, so duplicates remain\npossible. Forbid all slots of a selected label for each row to find the cheapest\ndistinct labeling. At most n+1 small assignment solves for n <= 18.\n"""\n    rows, classes = unary.shape\n    slot_labels = np.repeat(np.arange(classes), rows)\n    extra_cost = penalty * np.tile(np.arange(rows), classes)\n    cost = unary[:, slot_labels] + extra_cost[None, :]\n    row_index, slot_index = linear_sum_assignment(cost)\n    if not np.array_equal(row_index, np.arange(rows)):\n        raise AssertionError(\'Soft assignment failed to cover every group row\')\n    chosen = slot_labels[slot_index]\n    optimum = _soft_energy(unary, chosen, penalty)\n    if not np.isclose(float(cost[row_index, slot_index].sum()), optimum, atol=1e-9, rtol=0):\n        raise AssertionError(\'Repeated label slots disagree with the pairwise objective\')\n    second = np.inf\n    for row, label in enumerate(chosen):\n        alternative = cost.copy()\n        alternative[row, slot_labels == label] = np.inf\n        alt_rows, alt_slots = linear_sum_assignment(alternative)\n        second = min(second, float(alternative[alt_rows, alt_slots].sum()))\n    return chosen, optimum, max(0.0, float(second - optimum))\n\n\ndef soft_constraint_decode(probabilities, tfidf_groups, glm_groups, config=None):\n    """Return guarded MAP labels, retaining probabilities and rejected blocks intact."""\n    settings = dict(SOFT_MRF_DEFAULTS)\n    if config:\n        unknown = set(config) - set(settings)\n        if unknown:\n            raise ValueError(f\'Unknown soft-MRF settings: {sorted(unknown)}\')\n        settings.update(config)\n    for name in [\'penalty_strength\', \'min_energy_improvement\', \'min_configuration_gap\',\n                 \'max_mean_log_loss\']:\n        if not np.isfinite(settings[name]) or settings[name] < 0:\n            raise ValueError(f\'{name} must be finite and nonnegative\')\n    if settings[\'min_configuration_gap\'] <= 0:\n        raise ValueError(\'The configuration gap must be positive to reject tied labelings\')\n    for name in [\'min_group_confidence\', \'min_assigned_probability\', \'min_relative_probability\']:\n        if not 0 < settings[name] <= 1:\n            raise ValueError(f\'{name} must be in (0,1]\')\n    if (not isinstance(settings[\'min_group_size\'], int)\n            or not isinstance(settings[\'max_parent_size\'], int)\n            or not 2 <= settings[\'min_group_size\'] <= settings[\'max_parent_size\'] <= 18):\n        raise ValueError(\'Expected 2 <= min_group_size <= max_parent_size <= 18\')\n    if not isinstance(settings[\'max_assigned_rank\'], int) or settings[\'max_assigned_rank\'] < 1:\n        raise ValueError(\'The rank limit must be a positive integer\')\n    probability = np.asarray(probabilities, dtype=np.float64)\n    if (probability.ndim != 2 or len(probability) == 0 or probability.shape[1] < 2\n            or not np.isfinite(probability).all() or (probability < 0).any()\n            or not np.allclose(probability.sum(1), 1, atol=1e-6, rtol=0)):\n        raise ValueError(\'Expected finite, nonnegative, normalized probabilities\')\n    rows, classes = probability.shape\n    partitions = []\n    sizes = []\n    for name, values in [(\'tfidf\', tfidf_groups), (\'glm\', glm_groups)]:\n        values = np.asarray(values)\n        if values.shape != (rows,) or not np.issubdtype(values.dtype, np.integer):\n            raise ValueError(f\'{name} grouping must contain one integer ID per test row\')\n        _, inverse, counts = np.unique(values, return_inverse=True, return_counts=True)\n        partitions.append(inverse)\n        sizes.append(counts[inverse])\n    intersections, block_index = np.unique(np.column_stack(partitions), axis=0, return_inverse=True)\n    baseline = probability.argmax(1)\n    selected = baseline.copy()\n    decisions = []\n    for block in range(len(intersections)):\n        members = np.flatnonzero(block_index == block)\n        count = len(members)\n        parent_a, parent_b = int(sizes[0][members[0]]), int(sizes[1][members[0]])\n        confidence = count / (parent_a + parent_b - count)\n        penalty = float(settings[\'penalty_strength\'] * confidence)\n        record = {\'block\':int(block), \'rows\':int(count), \'tfidf_parent_rows\':parent_a,\n                  \'glm_parent_rows\':parent_b, \'confidence\':float(confidence),\n                  \'pair_penalty\':penalty, \'accepted\':False, \'changed_rows\':0}\n        if count < settings[\'min_group_size\']:\n            record[\'reason\'] = \'singleton_or_small_intersection\'\n        elif max(parent_a, parent_b) > settings[\'max_parent_size\']:\n            record[\'reason\'] = \'oversized_parent_group\'\n        elif confidence < settings[\'min_group_confidence\']:\n            record[\'reason\'] = \'low_grouping_confidence\'\n        elif penalty == 0:\n            record[\'reason\'] = \'zero_penalty\'\n        else:\n            old = baseline[members]\n            collisions = count - len(np.unique(old))\n            record[\'baseline_duplicate_labels\'] = int(collisions)\n            if not collisions:\n                record[\'reason\'] = \'no_label_collision\'\n                decisions.append(record)\n                continue\n            group_probability = probability[members]\n            unary = -np.log(np.clip(group_probability, 1e-12, 1))\n            candidate, energy, gap = _solve_soft_clique(unary, penalty)\n            baseline_energy = _soft_energy(unary, old, penalty)\n            improvement = baseline_energy - energy\n            if improvement < -1e-8:\n                raise AssertionError(\'Exact soft assignment increased the objective\')\n            assigned = group_probability[np.arange(count), candidate]\n            original = group_probability[np.arange(count), old]\n            relative = assigned / original\n            ranks = (group_probability > assigned[:, None]).sum(1) + 1\n            mean_loss = float(np.log(np.maximum(original, 1e-300)\n                                     / np.maximum(assigned, 1e-300)).mean())\n            changed = int(np.count_nonzero(candidate != old))\n            record.update({\'baseline_energy\':baseline_energy, \'candidate_energy\':energy,\n                           \'energy_improvement\':float(improvement), \'configuration_gap\':float(gap),\n                           \'min_assigned_probability\':float(assigned.min()),\n                           \'min_relative_probability\':float(relative.min()),\n                           \'max_assigned_rank\':int(ranks.max()), \'mean_log_loss\':mean_loss,\n                           \'candidate_changed_rows\':changed,\n                           \'candidate_duplicate_labels\':int(count - len(np.unique(candidate)))})\n            if not changed:\n                record[\'reason\'] = \'baseline_is_soft_optimum\'\n            elif improvement < settings[\'min_energy_improvement\']:\n                record[\'reason\'] = \'insufficient_energy_improvement\'\n            elif gap < settings[\'min_configuration_gap\']:\n                record[\'reason\'] = \'ambiguous_label_configuration\'\n            elif assigned.min() < settings[\'min_assigned_probability\']:\n                record[\'reason\'] = \'assigned_probability_too_low\'\n            elif relative.min() < settings[\'min_relative_probability\']:\n                record[\'reason\'] = \'individual_probability_loss\'\n            elif ranks.max() > settings[\'max_assigned_rank\']:\n                record[\'reason\'] = \'assigned_label_rank\'\n            elif mean_loss > settings[\'max_mean_log_loss\']:\n                record[\'reason\'] = \'group_probability_loss\'\n            else:\n                selected[members] = candidate\n                record.update({\'accepted\':True, \'reason\':\'accepted\', \'changed_rows\':changed})\n        decisions.append(record)\n    reasons = {}\n    for item in decisions:\n        reasons[item[\'reason\']] = reasons.get(item[\'reason\'], 0) + 1\n    report = {\'method\':\'weighted-cannot-link-mrf-exact-clique-map-v1\',\n              \'paper\':\'https://www.cs.utexas.edu/~ml/papers/semi-kdd-04.pdf\',\n              \'settings\':settings, \'rows\':int(rows), \'classes\':int(classes),\n              \'intersection_blocks\':int(len(intersections)),\n              \'accepted_blocks\':sum(int(item[\'accepted\']) for item in decisions),\n              \'changed_rows\':int(np.count_nonzero(selected != baseline)),\n              \'block_reasons\':reasons, \'thresholds_accuracy_validated\':False,\n              \'full_hmrf_kmeans_reproduction\':False, \'block_decisions\':decisions}\n    return selected, report\n'}
_NTL_PUBLIC_REFERENCE = {'ids': ['00353fb1c541', '005f5a9a6139', '0062a53d5487'], 'labels': ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'], 'probabilities': [[2.065235756597896e-08, 1.5402772200334827e-07, 1.7230889901935973e-08, 1.973224452265069e-08, 2.405796485293284e-08, 3.4723394535555245e-09, 7.088639603823157e-06, 2.3628434018824104e-07, 1.0193243770154625e-06, 5.7114259253191535e-05, 0.6739213430828271, 7.141375081932038e-09, 0.28954008162195494, 0.004499245504005391, 0.00015994382661630225, 0.023195513598242932, 0.005786206912891758, 0.0028319606309939326], [0.0723684379859632, 0.017783897350001562, 0.6143946431761461, 0.2948037400421579, 0.00017161804841888713, 5.836929544549067e-08, 6.105320454493134e-09, 6.469243181007317e-09, 1.6383476574527233e-08, 7.624665770594059e-10, 4.4335171518836765e-09, 5.724164046555114e-08, 1.502383991873213e-09, 4.755530872274465e-07, 0.0004170435996549914, 1.5771892852269086e-07, 4.067371512538571e-07, 5.9428521146331683e-05], [7.106295894606943e-11, 7.20722387488426e-10, 1.9918053682483456e-09, 1.057451058182039e-10, 1.0215390109138841e-10, 0.9923046052742458, 0.004866814877957657, 0.0027266625070061297, 0.00010151635825827161, 2.438215693305543e-07, 5.070958510431848e-08, 8.560486080105227e-10, 1.547476361050142e-09, 1.0601926174407002e-09, 4.533501082154746e-09, 2.3769104853410373e-09, 3.1544469435263822e-09, 8.993131202351023e-08]], 'submission_ref': 57564496, 'script_version_id': 361488750, 'score': 0.627, 'validation_only': True}
_NTL_EXPERIMENT = {'profile': 'public0627_small_encoders', 'kind': 'encoders', 'slug': 'ntl-0627-small-encoder-ensemble', 'title': 'NTL 0627 Small Encoder Ensemble', 'description': '0.627 plus 10% UltraBERT/ELECTRA mixture', 'needs_cache': True}
from pathlib import Path
import sys
for _name, _text in _NTL_EXTENSION_MODULES.items():
    Path(_name).write_text(_text, encoding='utf-8')
sys.path.insert(0, str(Path.cwd()))


# Reproduksi publik 0.627 — encoder 6:1

Sumber metode: [Pickle Rick, v2 / 361476098](https://www.kaggle.com/code/mrpicklerick/ntl-0-627-encoder-ratio-6-to-1?scriptVersionId=361476098),
[Nishant Dahal, two-stage LoRA part two](https://www.kaggle.com/code/nishantdahal/two-stage-lora-part-two),
dan [Nuri Dudak, encoder ensemble](https://www.kaggle.com/code/nuridudak/0-626-when-llm-decoders-meet-bert-encodersv2).

Resep inference sumber dipertahankan: dua LoRA 14B, DeBERTa, SVC, ModernBERT
0.30, Desklib 0.05, centered twins, Sinkhorn 300 iterasi, typography guards mati.
Semua checkpoint neural sudah dilatih; SVC difit pada CPU. T4 ×2, Internet OFF.

Reproduksi ini menjalankan model lengkap pada sampel publik tiga baris,
memeriksa file bobot dan label, menjaga ID sebagai string, dan menolak hasil
jika ada anggota ensemble yang dilewati. Angka 0.627 adalah skor sumber;
skor reproduksi ini ditentukan rerun submission Kaggle.


# Two-stage LoRA, part two, LB 0.620

Our [first notebook](https://www.kaggle.com/code/nishantdahal/two-stage-lora-on-the-public-pool) scored 0.592 with two Qwen3 LoRAs trained on 35,000 public passages. This one scores 0.620 with the same recipe. Three things changed. The public pool got bigger and cleaner, two new members joined, and fixed a tokenizer bug that only shows up on Kaggle. Below is what each change did, on the same 5-fold CV split by prompt.

| Member | Trains on | First notebook | This notebook |
|---|---|---|---|
| n-gram SVC | train plus the public pool, fit here | 0.435 | 0.473 |
| Qwen3-14B LoRA, 27 of 40 layers | public pool, then train | 0.569 | 0.611 |
| Qwen2.5-14B LoRA, 32 of 48 layers | public pool, then train | | 0.569 |
| DeBERTa-v3-large, full fine tune | public pool, then train | | 0.546 |
| Blend with Sinkhorn, CV | | 0.593 | **0.642** |
| Public LB, with the centred twin | | 0.592 | **0.620** |

Member rows are balanced accuracy of that member alone, out of fold. The first notebook also had a Qwen3-8B LoRA at 0.569, which is gone now.

Everything the notebook needs is in the dataset [NTL two-stage LoRA part two](https://www.kaggle.com/datasets/nishantdahal/ntl-handwriting-blend). The base models are Kaggle's copies of Qwen3 14B Base and Qwen2.5 14B. Inference only. The commit run on the public test takes about 15 minutes on two T4s.

In [ ]:
import gc, glob, json, os, re, shutil, time

# The scoring run has no internet, so transformers must never try to reach the Hub.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
from torch import nn

# Keep fp32 sums in fp16 matmuls. A T4 is otherwise allowed to sum in fp16, which drops digits in a deep model.
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

T0 = time.time()
INPUT_DIR = "/kaggle/input"

# Blend weights. Every member adds weight * log(probability) to the score.
W_SVC = 1.0
LORAS = {  # checkpoint folder, then blend weight, centred twin weight, batch size and the base model to look for
    "lora_qwen3_14b":  dict(w=0.4, twin_w=0.1, bs=8, base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
    "lora_qwen25_14b": dict(w=0.2, twin_w=0.1, bs=8, base=dict(model_type="qwen2", hidden_size=5120, num_hidden_layers=48)),
}
W_DEBERTA = 0.25
TWIN = dict(thresh=0.85, min_size=2, max_size=18)   # topic clusters that get the centred twin

# Column order of every checkpoint's head. Each checkpoint's meta.json is checked against it below.
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)


def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def vote(name, P, w, rows=slice(None)):
    """Add one member's weighted log probabilities to the running score."""
    LOGIT[rows] += w * np.log(np.clip(P, 1e-9, 1))
    print(f"{name}: weight {w}, done after {time.time() - T0:.0f}s", flush=True)


train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"), dtype={"id": str})
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map({l: i for i, l in enumerate(LABELS)}).values
LOGIT = np.zeros((len(test), K))
print("train", train.shape, "test", test.shape, "gpus", torch.cuda.device_count(), flush=True)

PUBLIC_SAMPLE = False  # exercise the complete ensemble on the public sample
if PUBLIC_SAMPLE:
    fallback = train["model"].mode().iat[0]
    pd.DataFrame({"id": test["id"].astype(str), "model": fallback}).to_parquet("submission.parquet", index=False)
    print("fast public validation output written; hidden scoring runs the full ensemble", flush=True)


In [ ]:
# Complementary encoder weights on the same log-probability scale as W_DEBERTA.
OURS = dict(modernbert=0.30, desklib=0.05)
TYPO_GUARDS = False


In [ ]:
_NTL_REPRODUCTION_CONTRACT = {'schema': 'ntl-public-0627-v1', 'source': {'kernel': 'mrpicklerick/ntl-0-627-encoder-ratio-6-to-1', 'version': 2, 'script_version_id': 361476098, 'public_score': 0.627, 'source_sha256': 'f224df58617bee5832047ad8e47bd4cf30a450ed7bbc66fb4f9c678aa0b933e8'}, 'mandatory_votes': {'svc': 1.0, 'lora_qwen3_14b': 0.4, 'lora_qwen25_14b': 0.2, 'deberta': 0.25, 'modernbert': 0.3, 'desklib': 0.05}, 'decoder_layers': {'lora_qwen3_14b': 27, 'lora_qwen25_14b': 32}, 'weights': {'deberta_v3_large/encoder/model.safetensors': {'bytes': 868069608, 'tensors': 390, 'shapes': {}}, 'deberta_v3_large/head.safetensors': {'bytes': 73936, 'tensors': 2, 'shapes': {'weight': [18, 1024], 'bias': [18]}}, 'lora_qwen25_14b/adapter/adapter_model.safetensors': {'bytes': 183558248, 'tensors': 448, 'shapes': {}}, 'lora_qwen25_14b/heads.safetensors': {'bytes': 6227664, 'tensors': 20, 'shapes': {'plain.weight': [18, 40960], 'centered.weight': [18, 40960]}}, 'lora_qwen3_14b/adapter/adapter_model.safetensors': {'bytes': 173456688, 'tensors': 378, 'shapes': {}}, 'lora_qwen3_14b/heads.safetensors': {'bytes': 6227664, 'tensors': 20, 'shapes': {'plain.weight': [18, 40960], 'centered.weight': [18, 40960]}}, 'desklib/model.safetensors': {'bytes': 870209556, 'tensors': 394, 'shapes': {'classifier.weight': [18, 1024], 'classifier.bias': [18]}}, 'modernbert/model.safetensors': {'bytes': 791717780, 'tensors': 174, 'shapes': {'classifier.weight': [18, 1024], 'classifier.bias': [18]}}}}
_NTL_REPRODUCTION_MODULES = {"public_0627_runtime.py": '"""Input and output checks for the credited public 0.627 reproduction."""\nfrom collections import Counter\nimport json\nfrom pathlib import Path\nimport time\n\n\ndef verify_reproduction_inputs(namespace, contract):\n    from safetensors import safe_open\n\n    if namespace[\'torch\'].cuda.device_count() != 2:\n        raise ValueError(\'This reproduction requires the two attached T4 GPUs\')\n    root = Path(namespace[\'INPUT_DIR\'])\n    labels = namespace[\'LABELS\']\n    if len(labels) != 18 or len(set(labels)) != 18:\n        raise ValueError(\'Expected the 18 competition labels\')\n    inspected = []\n    for relative, expected in contract[\'weights\'].items():\n        matches = sorted(root.rglob(relative))\n        if len(matches) != 1:\n            raise ValueError(f\'Expected one checkpoint file: {relative}; found {len(matches)}\')\n        path = matches[0]\n        if path.stat().st_size != expected[\'bytes\']:\n            raise ValueError(f\'Checkpoint size differs: {relative}\')\n        with safe_open(str(path), framework=\'pt\', device=\'cpu\') as checkpoint:\n            for key, shape in expected.get(\'shapes\', {}).items():\n                if checkpoint.get_slice(key).get_shape() != shape:\n                    raise ValueError(f\'Checkpoint head shape differs: {relative}:{key}\')\n            tensors = len(list(checkpoint.keys()))\n        if tensors != expected[\'tensors\']:\n            raise ValueError(f\'Checkpoint tensor count differs: {relative}\')\n        inspected.append({\'file\': relative, \'bytes\': path.stat().st_size, \'tensors\': tensors})\n\n    for name, spec in namespace[\'LORAS\'].items():\n        config = json.loads(Path(namespace[\'find\'](f\'{name}/meta.json\')).read_text())\n        if config[\'labels\'] != labels or config[\'cfg\'][\'eos_id\'] != 151643:\n            raise ValueError(f\'Label order or training EOS differs: {name}\')\n        if config[\'cfg\'][\'keep\'] != contract[\'decoder_layers\'][name]:\n            raise ValueError(f\'Decoder depth differs: {name}\')\n        # A matching config alone does not establish that base model shards exist.\n        candidates = []\n        for path in sorted(root.rglob(\'config.json\')):\n            config = json.loads(path.read_text())\n            if all(config.get(k) == v for k, v in spec[\'base\'].items()):\n                candidates.append(path.parent)\n        if len(candidates) != 1:\n            raise ValueError(f\'Expected one matching base model for {name}\')\n        base = candidates[0]\n        index = base / \'model.safetensors.index.json\'\n        shard_names = set(json.loads(index.read_text())[\'weight_map\'].values()) if index.exists() else {\'model.safetensors\'}\n        if not shard_names or any(not (base / shard).is_file() or (base / shard).stat().st_size == 0 for shard in shard_names):\n            raise ValueError(f\'Base model weights are incomplete: {name}\')\n\n    for name in [\'modernbert\', \'desklib\']:\n        config = json.loads(Path(namespace[\'find\'](f\'{name}/config.json\')).read_text())\n        mapping = config[\'id2label\']\n        if len(mapping) != 18 or set(mapping.values()) != set(labels):\n            raise ValueError(f\'Encoder class mapping differs: {name}\')\n        namespace[\'find\'](f\'{name}/tokenizer.json\')\n    if namespace[\'OURS\'] != {\'modernbert\': 0.30, \'desklib\': 0.05} or namespace[\'TYPO_GUARDS\']:\n        raise ValueError(\'The 0.627 encoder weights or typography setting changed\')\n    report = {\'passed\': True, \'weight_files\': inspected, \'base_models_present\': 2,\n              \'full_public_inference\': not namespace[\'PUBLIC_SAMPLE\'],\n              \'ids\': namespace[\'test\'][\'id\'].astype(str).tolist(),\n              \'labels\': labels, \'source\': contract[\'source\']}\n    Path(\'reproduction-inputs.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    print(\'Verified eight real checkpoint files, two base models and all 18 labels\', flush=True)\n    return report\n\n\ndef trace_reproduction_votes(namespace):\n    import numpy as np\n\n    original = namespace[\'vote\']\n    trace = []\n\n    def checked_vote(name, probabilities, weight, rows=slice(None)):\n        values = np.asarray(probabilities)\n        expected_rows = len(namespace[\'LOGIT\'][rows])\n        if values.shape != (expected_rows, len(namespace[\'LABELS\'])):\n            raise ValueError(f\'Invalid probability shape for {name}\')\n        if not np.isfinite(values).all() or (values < 0).any() or not np.allclose(values.sum(1), 1):\n            raise ValueError(f\'Invalid probabilities for {name}\')\n        original(name, probabilities, weight, rows=rows)\n        trace.append({\'name\': name, \'weight\': float(weight), \'rows\': expected_rows})\n\n    namespace[\'vote\'] = checked_vote\n    return trace\n\n\ndef validate_reproduction_result(namespace, contract, trace):\n    import numpy as np\n\n    rows = len(namespace[\'test\'])\n    labels = namespace[\'LABELS\']\n    required = contract[\'mandatory_votes\']\n    counts = Counter(item[\'name\'] for item in trace)\n    if any(counts[name] != 1 for name in required):\n        raise ValueError(\'A required ensemble member was skipped or ran more than once\')\n    for item in trace:\n        name = item[\'name\']\n        expected = required.get(name, 0.1 if name.endswith(\' twin\') else None)\n        if expected is None or item[\'weight\'] != expected:\n            raise ValueError(f\'Unexpected ensemble vote: {name}\')\n        if name in required and item[\'rows\'] != rows:\n            raise ValueError(f\'An ensemble member omitted rows: {name}\')\n    twin_rows = int(namespace[\'twin_rows\'].sum())\n    for decoder in contract[\'decoder_layers\']:\n        twin_name = decoder + \' twin\'\n        if counts[twin_name] != int(twin_rows > 0):\n            raise ValueError(f\'Centered twin was omitted or unexpectedly added: {decoder}\')\n        if any(item[\'rows\'] != twin_rows for item in trace if item[\'name\'] == twin_name):\n            raise ValueError(f\'Centered twin row count differs: {decoder}\')\n    probabilities = np.asarray(namespace[\'P\'])\n    if probabilities.shape != (rows, 18) or not np.isfinite(probabilities).all():\n        raise ValueError(\'Invalid final probability matrix\')\n    if (probabilities < 0).any() or not np.allclose(probabilities.sum(1), 1):\n        raise ValueError(\'Final probability rows do not sum to one\')\n    result = namespace[\'sub\']\n    expected_ids = namespace[\'test\'][\'id\'].astype(str).tolist()\n    if list(result.columns) != [\'id\', \'model\'] or result[\'id\'].tolist() != expected_ids:\n        raise ValueError(\'Submission columns or row IDs differ from test.csv\')\n    if not result[\'id\'].is_unique or not result[\'model\'].isin(labels).all():\n        raise ValueError(\'Submission IDs or class labels are invalid\')\n    if result[\'model\'].tolist() != np.asarray(labels)[probabilities.argmax(1)].tolist():\n        raise ValueError(\'Submission labels differ from the final ensemble\')\n    if namespace[\'PUBLIC_SAMPLE\'] or namespace[\'TYPO_GUARDS\']:\n        raise ValueError(\'Unexpected public fallback or typography guards\')\n    report = {\'pipeline\': \'public-0627-reproduction-v1\', \'passed\': True,\n              \'rows\': rows, \'ids\': expected_ids, \'labels\': labels, \'votes\': trace,\n              \'mandatory_members_completed\': list(required), \'full_public_inference\': True,\n              \'sinkhorn_applied\': rows >= 50,\n              \'centered_twin_rows\': twin_rows,\n              \'qwen_or_encoder_training\': False, \'svc_fit_on_cpu\': True,\n              \'typography_guards\': False, \'source\': contract[\'source\'],\n              \'seconds\': time.time() - namespace[\'T0\'], \'score\': None}\n    return report\n'}
exec(compile(_NTL_REPRODUCTION_MODULES["public_0627_runtime.py"], "public_0627_runtime.py", "exec"), globals())
_NTL_INPUT_REPORT = verify_reproduction_inputs(globals(), _NTL_REPRODUCTION_CONTRACT)
_NTL_VOTE_TRACE = trace_reproduction_votes(globals())


## 1. n-gram SVC

The same four TF-IDF views as the first notebook. Raw characters, words with punctuation, word shapes, and small words with every content word replaced by CONTENT. The same LinearSVC with C of 0.5 and balanced class weights. The only change is the pool it fits on, 75,783 public passages instead of 34,940. That alone moved it from 0.435 to 0.473.

In [ ]:
if not PUBLIC_SAMPLE:
    from scipy.sparse import hstack
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.svm import LinearSVC

    SMALL_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
    VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("small", 1, 3, 50000, 0.5)]


    def shape_text(t):
        """Every run of letters becomes X or x, so only the rhythm is left."""
        return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)


    def small_words_text(t):
        """Keep the small words and the punctuation, and replace every other word by CONTENT."""
        return " ".join(w.lower() if w.lower() in SMALL_WORDS or not w.isalnum() else "CONTENT"
                        for w in re.findall(r"\w+|[^\w\s]", t))


    pool = pd.read_parquet(find("pool_v4.parquet"))
    fit_texts = train.text.astype(str).tolist() + pool.text.tolist()
    fit_y = np.r_[y, pool.model.map({l: i for i, l in enumerate(LABELS)}).values]
    A, B = [], []
    for view, lo, hi, max_features, weight in VIEWS:
        prep = shape_text if view == "shape" else small_words_text if view == "small" else (lambda s: s)
        analyzer = "char" if view == "shape" else "word" if view == "small" else view
        opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                    max_features=max_features, dtype=np.float32)
        if analyzer == "word":
            opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
        vec = TfidfVectorizer(**opts)
        A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)
        B.append(vec.transform([prep(t) for t in test.text]) * weight)
    svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(hstack(A, format="csr"), fit_y)
    vote("svc", softmax(svc.decision_function(hstack(B, format="csr"))), W_SVC)
    del A, B, svc, pool
    gc.collect()

## 2. The pool, the change that mattered most

Most of the gain came from data, not from models. On the old pool of 35,000 passages, four backbones finished at the same 0.569. Qwen3-8B, Qwen3-14B, Qwen2.5-7B and Qwen2.5-14B. The same Qwen3-14B with the same recipe on the new pool finished at 0.611.

You can see it before the host data even comes in. After stage one alone, trained only on public passages, the 14B's head scores 0.572 on the host passages. With the old pool it scored 0.493. The backbone did not change. The pool did.

The pool the 14B read in stage one has 83,827 passages from 26 public datasets. It keeps the filters of the first notebook. One paragraph, 30 to 300 words, no markdown, no duplicates, nothing identical to a train passage, and no punctuation mark that a model never uses in the host data. Two filters are new.

- **One source is out.** A set of claude-fable-5 passages written with a system prompt. They read nothing like the host's Fable passages.
- **Public passages too close to a train passage are out.** 1,153 passages whose content TF-IDF cosine with some train passage is 0.3 or more. Some public datasets answer prompts rebuilt from the training data. If stage one has already seen a host prompt, the out of fold score of stage two is too kind. This filter keeps the CV honest. It does not raise it.

The dataset ships the 75,783 passage version that the SVC fits on. It has the first filter but not the second.

## 3. Two new members

**Qwen2.5-14B LoRA.** The same recipe as the Qwen3 LoRAs, on the first 32 of 48 layers, pooled at layers 3, 5, 8, 11, 16, 21, 27 and 32. Stage one ran on the old 35,000 passage pool. Alone it ties the Qwen3 models at 0.569. Its top prediction agrees with theirs on about 77 percent of passages, which is about as often as they agree with each other. So it is not a new view, and it gets a small weight, 0.2.

**DeBERTa-v3-large, fully fine tuned.** No LoRA here, all the weights train. The same two stages. Stage one is 2 epochs over 61,486 public passages with class weights, at learning rate 0.000015 for the encoder and 0.001 for the head. Stage two is 3 epochs on train at 0.00001 and 0.0005. Max length 384, the mean of the last layer, one linear head. The bigger pool moved it from 0.512 to 0.546 alone.

## 4. LoRA inference, and a bug that only shows up on Kaggle

The inference code is the first notebook's. Keep the first layers, merge the adapter into the weights, average the hidden states at 8 depths without the first token, one LayerNorm per depth, one linear head. Two things are new.

**The end token.** Every passage gets an end token appended, in training and at inference. Our training tokenizer used `<|endoftext|>`, id 151643. Kaggle's copy of Qwen3-14B-Base ships the tokenizer config of the chat model, whose end token is `<|im_end|>`, id 151645. So on Kaggle every passage ended with a token the model never saw there in training, and the 14B's top prediction changed on about 14 percent of passages compared with the same checkpoint run locally. Nothing crashes and nothing warns. The fix is one line. Use the id saved in the checkpoint's meta.json. Kaggle's Qwen2.5-14B does not have this problem. If you run Qwen3-14B-Base from Kaggle Models, print which end token you get.

**T4 numerics and memory.**

- We tell cuBLAS to keep fp32 sums in fp16 matmuls. It costs nothing, so we set it.
- A batch that overflows fp16 is redone in bf16, as before.
- A batch that runs out of memory on the 15 GB T4 is redone one passage at a time.
- There is no try except block around the members. In an earlier submission a handler like that skipped a member that ran out of memory on the T4. The run still finished and scored, a few points lower, and we only found out from the log. A member that breaks should stop the notebook.

In [ ]:
if not PUBLIC_SAMPLE:
    from safetensors.torch import load_file
    from transformers import AutoConfig, AutoModel, AutoTokenizer


    def find_base(spec):
        """The attached base model whose config.json matches spec."""
        for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
            try:
                c = json.load(open(cfg))
            except Exception:
                continue
            if all(c.get(k) == v for k, v in spec.items()):
                return os.path.dirname(cfg)
        raise FileNotFoundError(f"attach the base model {spec}")


    def load_trunk(base_dir, keep, dtype):
        """The first keep decoder layers, split over both T4s when they do not fit on one."""
        mc = AutoConfig.from_pretrained(base_dir)
        mc.num_hidden_layers, mc.use_cache = keep, False
        if getattr(mc, "layer_types", None) is not None:
            mc.layer_types = mc.layer_types[:keep]
        H, I, V = mc.hidden_size, mc.intermediate_size, mc.vocab_size
        hd = getattr(mc, "head_dim", H // mc.num_attention_heads)
        q, kv = mc.num_attention_heads * hd, mc.num_key_value_heads * hd
        gib = 2 * (V * H + keep * (2 * H * q + 2 * H * kv + 3 * H * I)) / 2 ** 30
        if gib > 13 and torch.cuda.device_count() > 1:
            mem = {i: "13GiB" for i in range(torch.cuda.device_count())}
            model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()
        else:
            model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()
        model.norm = nn.Identity()   # the checkpoints were trained without the final norm
        print(f"trunk of {keep} layers, {gib:.1f} GiB of weights", flush=True)
        return model


    @torch.no_grad()
    def merge_lora(model, adapter_dir):
        """W becomes W + B A alpha / r for every adapted linear layer."""
        acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
        scale = acfg["lora_alpha"] / acfg["r"]
        sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
        modules = dict(model.named_modules())
        for k, A in sd.items():
            if ".lora_A." not in k:
                continue
            name = k.split(".lora_A.")[0]
            lin = modules[name[name.index("layers."):]]
            Bm = sd[k.replace(".lora_A.", ".lora_B.")]
            delta = (Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale
            lin.weight.add_(delta.to(lin.weight.dtype))


    @torch.no_grad()
    def lora_features(ckpt_dir, base_dir, texts, bs):
        """Pooled features Z, one row per passage, and the checkpoint's heads."""
        meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))
        assert meta["labels"] == LABELS, "label order differs from the checkpoint's heads"
        cfg = meta["cfg"]
        dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        tok = AutoTokenizer.from_pretrained(base_dir)
        tok.padding_side = "right"
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        model = load_trunk(base_dir, cfg["keep"], dtype)
        merge_lora(model, os.path.join(ckpt_dir, "adapter"))
        heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))
        layers, H = cfg["layers"], model.config.hidden_size
        norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
        norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
        ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]
        # The end token from training, not this tokenizer's. Kaggle's Qwen3-14B-Base carries the chat model's end token.
        eos_id = cfg.get("eos_id", tok.eos_token_id)
        print(f"end token {eos_id}, this tokenizer's is {tok.eos_token_id}", flush=True)
        ids = [x + [eos_id] for x in ids]
        order = np.argsort([len(x) for x in ids], kind="stable")   # similar lengths together means less padding
        Z = np.zeros((len(ids), H * len(layers)), np.float32)

        def embed(idx):
            enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow, so redo this batch in bf16
                model.to(torch.bfloat16)
                hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
                model.to(dtype)
            m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
            m[:, 0] = 0   # the first token soaks up attention whatever it is, so leave it out of the average
            den = m.sum(1).clamp(min=1)
            Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()

        for s in range(0, len(order), bs):
            idx = order[s:s + bs]
            try:
                embed(idx)
            except torch.cuda.OutOfMemoryError:   # the longest batch can overflow a 15 GB T4, so go one passage at a time
                torch.cuda.empty_cache()
                for i in idx:
                    embed([i])
        del model
        gc.collect()
        torch.cuda.empty_cache()
        return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}

## 5. Centred twin

As in the first notebook. Cluster the test passages by content TF-IDF. In every cluster of 2 to 18 passages, subtract the cluster mean from each LoRA's features and read the rest with the centred head, at weight 0.1 per LoRA. It is not in the CV numbers above, because CV has no test clusters.

In [ ]:
if not PUBLIC_SAMPLE:
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform

    if len(test) >= 2:
        Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test.text.tolist())
        D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
        np.fill_diagonal(D, 0)
        groups = fcluster(linkage(squareform(D, checks=False), method="average"), TWIN["thresh"], criterion="distance")
        del D
    else:
        groups = np.ones(len(test), dtype=int)
    size = np.bincount(groups)[groups]
    twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
    print(f"topic clusters: {int((size == 1).sum())} passages alone, {int(twin_rows.sum())} in clusters of 2 to 18", flush=True)

    for ckpt_name, mc in LORAS.items():
        ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))
        Z, heads = lora_features(ckpt, find_base(mc["base"]), test.text.tolist(), mc["bs"])
        vote(ckpt_name, softmax(Z @ heads["plain.weight"].T + heads["plain.bias"]), mc["w"])
        if twin_rows.any():
            Zc = Z.copy()
            for g in np.unique(groups[twin_rows]):
                Zc[groups == g] -= Zc[groups == g].mean(0)
            centred = Zc[twin_rows] @ heads["centered.weight"].T + heads["centered.bias"]
            vote(ckpt_name + " twin", softmax(centred), mc["twin_w"], rows=twin_rows)
        del Z
        gc.collect()

## 6. DeBERTa

The checkpoint is copied to local disk before loading. Loading it straight from the dataset mount took 16 minutes in our runs. The copy plus the load take 25 seconds.

In [ ]:
if not PUBLIC_SAMPLE:
    ck = os.path.dirname(find("deberta_v3_large/meta.json"))
    if ck.startswith("/kaggle/input"):   # one long copy to local disk beats many small reads from the dataset mount
        shutil.copytree(ck, "/tmp/deberta_v3_large", dirs_exist_ok=True)
        ck = "/tmp/deberta_v3_large"
    meta = json.load(open(os.path.join(ck, "meta.json")))
    assert meta["labels"] == LABELS, "label order differs from the checkpoint's head"
    etok = AutoTokenizer.from_pretrained(os.path.join(ck, "encoder"))
    enc = AutoModel.from_pretrained(os.path.join(ck, "encoder"), dtype=torch.float32).to("cuda:0").eval()
    head = load_file(os.path.join(ck, "head.safetensors"))
    W, b = head["weight"].float().numpy(), head["bias"].float().numpy()
    ids = etok(test.text.tolist(), truncation=True, max_length=meta["max_len"])["input_ids"]
    order = np.argsort([len(x) for x in ids], kind="stable")
    Z = np.zeros((len(ids), enc.config.hidden_size), np.float32)
    with torch.inference_mode():
        for s in range(0, len(order), 32):
            idx = order[s:s + 32]
            bt = etok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to("cuda:0")
            m = bt["attention_mask"].unsqueeze(-1).float()
            Z[idx] = ((enc(**bt).last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
    vote("deberta", softmax(Z @ W.T + b), W_DEBERTA)
    del enc
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
_NTL_ENCODER_MODULES = {"enc_infer.py": '"""One encoder on one T4: fp32 weights + fp16 autocast (T4 tensor cores), length-sorted dynamic padding,\nfp32 retry for any batch that overflows in fp16. Writes raw logits + the head\'s label order."""\nimport sys, json, os, time, numpy as np, pandas as pd, torch\nfrom transformers import AutoTokenizer, AutoModelForSequenceClassification\npath, data, out = sys.argv[1:4]; t0 = time.time()\ntest = pd.read_csv(data, dtype={"id": str}); texts = test.text.fillna(\'\').astype(str).tolist()\ntok = AutoTokenizer.from_pretrained(path)\nmodel = AutoModelForSequenceClassification.from_pretrained(path, dtype=torch.float32).cuda().eval()\nlabels = [model.config.id2label[i] for i in range(model.config.num_labels)]\nids = tok(texts, truncation=True, max_length=512)[\'input_ids\']; order = np.argsort([len(x) for x in ids])\nlogits = np.zeros((len(texts), len(labels)), np.float32); retried = 0\nwith torch.inference_mode():\n    for s in range(0, len(order), 64):\n        idx = order[s:s + 64]; b = tok.pad({\'input_ids\': [ids[i] for i in idx]}, return_tensors=\'pt\').to(\'cuda\')\n        with torch.autocast(\'cuda\', dtype=torch.float16):\n            lg = model(**b).logits.float()\n        if not torch.isfinite(lg).all():\n            lg = model(**b).logits.float(); retried += 1\n        logits[idx] = lg.cpu().numpy()\nassert np.isfinite(logits).all()\nnp.savez(out, logits=logits, labels=np.array(labels, dtype=str), ids=test[\'id\'].astype(str).to_numpy(dtype=str))   # unicode, not object\nprint(f\'{os.path.basename(path)}: {len(texts)} rows in {time.time() - t0:.0f}s, fp32 retries {retried}\', flush=True)'}
open("enc_infer.py", "w", encoding="utf-8").write(_NTL_ENCODER_MODULES["enc_infer.py"])


In [ ]:
from ntl_staging import stage_checkpoint as _ntl_stage_checkpoint
if not PUBLIC_SAMPLE:
    ## our encoder members: both in parallel, one per T4, in child processes (the base members have released the GPUs)
    import subprocess, sys
    torch.cuda.empty_cache()
    try:
        OURS_ROOT = os.path.dirname(find("light_members.npz"))
    except FileNotFoundError:
        OURS_ROOT = None; print("our members SKIPPED: dataset nuridudak/ntl-ours-v1 not attached")
    jobs = {}
    for gpu, name in enumerate([n for n, w in OURS.items() if w > 0 and OURS_ROOT]):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu % max(1, torch.cuda.device_count())))
        jobs[name] = subprocess.Popen([sys.executable, "enc_infer.py", str(_ntl_stage_checkpoint(os.path.join(OURS_ROOT, name))), find("test.csv"), f"/kaggle/working/{name}_logits.npz"],
                                      env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for name, job in jobs.items():
        out, _ = job.communicate()
        print("\n".join(l for l in out.splitlines() if "arn" not in l)[-1500:], flush=True)
        try:
            assert job.returncode == 0, f"exit code {job.returncode}"
            z = np.load(f"/kaggle/working/{name}_logits.npz")
            assert list(z["ids"]) == test["id"].astype(str).tolist(), "row order differs"
            lab = [str(l) for l in z["labels"]]
            Pm = softmax(z["logits"][:, [lab.index(l) for l in LABELS]].astype(np.float64))
            assert np.isfinite(Pm).all()
            vote(name, Pm, OURS[name])
        except Exception as e:
            print(f"{name} SKIPPED: {e!r}", flush=True)

## 7. Blend and Sinkhorn

A weighted sum of log probabilities. SVC 1.0, Qwen3-14B 0.4, Qwen2.5-14B 0.2, DeBERTa 0.25. These are round numbers chosen on out of fold predictions. Fitting them finer on 2,699 passages mostly fits noise. Then Sinkhorn balancing to equal class totals as in the first notebook, 300 iterations, skipped on the 3 row public test.

In [ ]:
if not PUBLIC_SAMPLE:
    def sinkhorn(P, iters=300):
        """One scale per class, so that every class gets an equal share of the probability. Rows are renormalised."""
        c = np.ones(K)
        target = np.full(K, len(P) / K)
        for _ in range(iters):
            R = P * c
            R /= R.sum(1, keepdims=True)
            c *= target / R.sum(0)
        R = P * c
        return R / R.sum(1, keepdims=True)


    P = softmax(LOGIT)
    if len(test) >= 50:   # skip on the 3 row public test.csv
        P = sinkhorn(P)
    if TYPO_GUARDS:   # our typography guards: act only when the current top label is impossible for a mark in the passage
        RULES = [(" \u2014 ", "allowed", {"claude-opus-4-6", "claude-opus-5", "glm-5.3", "glm-5.3-flash"}),
                 ("\u2019", "forbidden", {l for l in LABELS if l.startswith(("claude", "glm"))}),
                 ('"', "forbidden", {"gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra", "qwen3.8-max"})]
        P = P.copy()
        for mark, kind, S in RULES:
            bad = np.array([(l not in S) if kind == "allowed" else (l in S) for l in LABELS]); n = 0
            for i, t in enumerate(test.text.values):
                if mark in t and bad[P[i].argmax()]:
                    P[i, bad] = 0.0; n += 1
            print(f"guard {mark!r}: {n} rows changed", flush=True)
    sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})
    assert len(sub) == len(test) and sub["model"].isin(LABELS).all()
    sub.to_parquet("submission.parquet", index=False)
    print(sub["model"].value_counts().to_string())
    print(f"done in {time.time() - T0:.0f}s")

## CV says 0.642, LB says 0.620

The first notebook lost almost nothing between CV and LB, 0.593 against 0.592. This one loses 2.2 points. What changed between the two is mostly the pool. Our CV only tests on host passages, but the members learned much of what they know from public passages written with other settings. Some of that seems not to carry over to the hidden test. We have guesses and no clean answer yet. Part of the gap is plain noise too, since the public LB is only a slice of the test.

## What did not help

- **More parameters.** See section 2.
- **Averaging seeds**, or averaging the LoRA weights of several runs. 0.1 points at most.
- **A Gemma-4-12B backbone** with the same recipe.
- **Gates for one family**, for example a separate judge just for the Claude and GLM passages.
- **At most one passage per model inside a topic cluster.** Better in simulation, worse on the LB.
- **More stage two epochs**, or public passages mixed into stage two.
- **Stage two on short windows** cut from the host passages. Short passages are hard for every member, and the windows did not change that.
- **A stacker** instead of fixed weights.

None of it helped the two hardest pairs, `glm-5.3` against `glm-5.3-flash` and `gemini-3.7-flash` against `gemini-3.8-flash`. Every member gets stuck at about the same place on them.

## Credits

Public passages, CC BY 4.0 unless another license is given.

- [seyitkaangunes/ntl-extra-api-passages](https://www.kaggle.com/datasets/seyitkaangunes/ntl-extra-api-passages)
- [willrice/name-that-llm-openrouter-harvest](https://www.kaggle.com/datasets/willrice/name-that-llm-openrouter-harvest)
- [lxdlxdlxd/ntl-synth-generations-v1](https://www.kaggle.com/datasets/lxdlxdlxd/ntl-synth-generations-v1)
- [prvsiyan/prvsiyans-extended-name-that-llm-dataset](https://www.kaggle.com/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset)
- [alperen5252525/ntl-vertex-gemini-proxy-gpt-generations](https://www.kaggle.com/datasets/alperen5252525/ntl-vertex-gemini-proxy-gpt-generations)
- [haileycheng/name-that-llm-same-prompt-passages](https://www.kaggle.com/datasets/haileycheng/name-that-llm-same-prompt-passages)
- [kaplinskiy/ntl-gpt-proxy-passages](https://www.kaggle.com/datasets/kaplinskiy/ntl-gpt-proxy-passages)
- [lucasboesen/ntl-free-proxy-generations](https://www.kaggle.com/datasets/lucasboesen/ntl-free-proxy-generations)
- [abdullah1afify/ntl-generations](https://www.kaggle.com/datasets/abdullah1afify/ntl-generations)
- [dophuchung/ntl-proxy-generations](https://www.kaggle.com/datasets/dophuchung/ntl-proxy-generations)
- [ziyuzhou1207/ntl-zz-proxy-generations](https://www.kaggle.com/datasets/ziyuzhou1207/ntl-zz-proxy-generations)
- [keshavatearth/ntl-gpt-new-prompts-v1](https://www.kaggle.com/datasets/keshavatearth/ntl-gpt-new-prompts-v1)
- [michaelsemenoff/ntl-own-extra-v1](https://www.kaggle.com/datasets/michaelsemenoff/ntl-own-extra-v1)
- [jadkutbi/ntl-own-proxy-generations](https://www.kaggle.com/datasets/jadkutbi/ntl-own-proxy-generations)
- [pardheev/ntl-proxy-extra-passages](https://www.kaggle.com/datasets/pardheev/ntl-proxy-extra-passages)
- [abhinavsaipodugu/ntl-proxy-gpt-gemini-passages](https://www.kaggle.com/datasets/abhinavsaipodugu/ntl-proxy-gpt-gemini-passages)
- [zakirkhanaleemi/name-that-llm-benchmarks-proxy-generations](https://www.kaggle.com/datasets/zakirkhanaleemi/name-that-llm-benchmarks-proxy-generations)
- [zbigniewkirchner/ntl-extra-gpt-passages](https://www.kaggle.com/datasets/zbigniewkirchner/ntl-extra-gpt-passages)
- [arxitekton/ntl-claude-gen-v1](https://www.kaggle.com/datasets/arxitekton/ntl-claude-gen-v1)
- [arxitekton/ntl-claude-gen-v2](https://www.kaggle.com/datasets/arxitekton/ntl-claude-gen-v2)
- [maniceet/name-that-llm-openrouter-openai-generations](https://www.kaggle.com/datasets/maniceet/name-that-llm-openrouter-openai-generations)
- [dedruck/ntl-proxy-gen](https://www.kaggle.com/datasets/dedruck/ntl-proxy-gen)
- [prvsiyan/ntl-own-prompt-generations-v1](https://www.kaggle.com/datasets/prvsiyan/ntl-own-prompt-generations-v1), CC BY-SA 4.0
- [drjcfmd/ntl-generated-v2](https://www.kaggle.com/datasets/drjcfmd/ntl-generated-v2), CC BY-SA 4.0
- [illiadolenko/ntl-generated-corpus-v1](https://www.kaggle.com/datasets/illiadolenko/ntl-generated-corpus-v1), CC0
- [aswathvanan/ntl-proxy-passages](https://www.kaggle.com/datasets/aswathvanan/ntl-proxy-passages), CC BY-SA 4.0, only in stage one of the Qwen3-14B LoRA

Ideas and recipes.

- The LoRA24 recipe from [lightningv08/ntl-lora24-fivefold-training](https://www.kaggle.com/code/lightningv08/ntl-lora24-fivefold-training) and [lucasboesen/name-that-llm-train-lora24-s7](https://www.kaggle.com/code/lucasboesen/name-that-llm-train-lora24-s7)
- The style aware n-gram views from [yangkuangou](https://www.kaggle.com/yangkuangou)'s notebook
- Adaptive centering, the centred twin and Sinkhorn balancing from [lucasboesen/lora-x-adaptive-centering](https://www.kaggle.com/code/lucasboesen/lora-x-adaptive-centering)

Models. Qwen3-14B-Base and Qwen2.5-14B by the Qwen team, Apache 2.0. DeBERTa-v3-large by Microsoft, MIT.

In [ ]:
_NTL_REPORT = validate_reproduction_result(globals(), _NTL_REPRODUCTION_CONTRACT, _NTL_VOTE_TRACE)
import importlib.metadata as _ntl_package_metadata
_NTL_REPORT["versions"] = {name: _ntl_package_metadata.version(name) for name in ["numpy", "scipy", "scikit-learn", "torch", "transformers", "safetensors"]}
_NTL_REPORT["inputs_verified"] = _NTL_INPUT_REPORT["passed"]
open("reproduction-report.json", "w", encoding="utf-8").write(json.dumps(_NTL_REPORT, indent=2))
np.savez_compressed("reproduction-probabilities.npz", probabilities=P, logit=LOGIT, ids=test["id"].astype(str).to_numpy(dtype=str), labels=np.array(LABELS, dtype=str))
print("REPRODUCTION VALIDATED: all six required members completed", flush=True)

from ntl_encoder_experiments import run_variant
_NTL_VARIANT_REPORT = run_variant(globals(), _NTL_EXPERIMENT)
